# develop and run sim

## libraries

In [4]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import scipy
import numpy as np
from pprint import pprint
import pickle
import heapq
from collections import deque
from tqdm import tqdm

## import network data

In [5]:
with open(r'.\data\blue_network.pkl', 'rb') as f:
    blue_network = pickle.load(f)

In [6]:
pprint(blue_network)

{'Airport': {False: {'dwell_params': {'c': 7.22,
                                      'd': 12.6,
                                      'loc': 0.0,
                                      'scale': 32.45},
                     'next_station': {'name': 'Maverick',
                                      'travel_params': {'c': 15.46,
                                                        'd': 4.01,
                                                        'loc': 0.0,
                                                        'scale': 88.74}}},
             True: {'dwell_params': {'c': 7.75,
                                     'd': 509.18,
                                     'loc': 0.0,
                                     'scale': 20.01},
                    'next_station': {'name': 'Wood Island',
                                     'travel_params': {'c': 16.22,
                                                       'd': 7.48,
                                                       'loc': 0.0,


## Discrete Event Simulation

In [7]:
class Train:
    """Represents a single train in the simulation, tracking its position,
    direction, and arrival/departure times at each station."""

    def __init__(self, train_id, station, direction, departure_time, network):
        self.train_id = train_id
        self.station = station                # current station name
        self.direction = direction            # True=outbound, False=inbound
        self.departure_time = departure_time  # scheduled initial departure time
        self.headway = []                     # list of headways recorded at each departure
        self.arrival   = {station_name: [] for station_name in network}   # arrival times per station
        self.departure = {station_name: [] for station_name in network}   # departure times per station


class Sim:
    """Discrete-event simulation of a metro line.
    
    Models trains circulating around a loop, sampling travel and dwell times
    from fitted distributions. Collects station-level statistics, timetables,
    and headway data across multiple replications.

    Parameters
    ----------
    network     : dict — station network generated by `generate_stations()`
    until       : int  — simulation end time in seconds
    distribution: scipy.stats distribution — used to sample travel/dwell times
    dispatch    : list — list of (time, station, direction) tuples
    repetition  : int  — number of simulation replications
    loop_time   : int  — total cycle time in seconds (default 2400s = 40 min)
    seed        : int or None — random seed for reproducibility (default None)
    rush_hour   : list[dict] or None — time windows with parameter scalars, e.g.:
                  [
                    {'from': 7200, 'to': 14400, 'params': {'scale': 1.3}},
                    {'from': 57600, 'to': 64800, 'params': {'scale': 1.2}},
                  ]
                  'from'/'to' are clock seconds; 'params' maps param name → scalar multiplier.
                  First matching window is applied; no window = no scaling.
    """

    def __init__(self, network, until, distribution, dispatch, repetition=1000,
                 loop_time=2400, seed=None, rush_hour=None):
        self.network      = network
        self.until        = until
        self.distribution = distribution
        self.dispatch     = dispatch
        self.repetition   = repetition
        self.loop_time    = loop_time
        self.rng          = np.random.default_rng(seed)
        self.rush_hour    = rush_hour or []
        self.station_df   = pd.DataFrame()   # accumulated station stats across replications
        self.timetable    = pd.DataFrame()   # accumulated timetable across replications
        self.train_headway = pd.DataFrame()  # accumulated headway data across replications

    def sample(self, **params):
        """Sample a travel or dwell time from the fitted distribution.
        Scales distribution parameters by rush_hour scalars if the current
        clock falls within a defined window. Returns at least 1 second."""
        if self.rush_hour:
            for window in self.rush_hour:
                if window['from'] <= self.clock < window['to']:
                    params = {k: v * window['params'].get(k, 1) for k, v in params.items()}
                    break  # first matching window wins
        return max(1, round(self.distribution(**params).rvs(random_state=self.rng)))

    def fresh_state(self):
        """Reset simulation state for a new replication —
        clears clock, event calendar, trains, and station state."""
        self.clock = 0
        self.event_calendar = []
        self.trains = {}  # train_id -> Train

        self.station_state = {}
        for station in self.network:
            self.station_state[station] = {
                True:  self._empty_station_stats(),
                False: self._empty_station_stats(),
            }

    def _empty_station_stats(self):
        """Initialise blank statistics dict for a station-direction pair."""
        return {
            'queue':          deque(),  # queue of (arrival_time, train_id) tuples
            'occupied_by':    None,     # train_id currently at platform (None if free)
            'visit':          0,        # number of trains served
            'total_arrivals': 0,        # total trains that arrived (including queued)
            'wait_time':      0,        # cumulative platform wait time
            'queue_weighted': 0,        # queue length weighted by wait time (for avg queue length)
            'last_arrival':   0,        # clock time of last arrival
            'last_departure': 0,        # clock time of last departure
            'frequency':      0,        # cumulative inter-departure time (for avg headway)
        }

    def push(self, event_time, event_type, train_id):
        """Push a new event onto the priority queue (min-heap by event_time)."""
        heapq.heappush(self.event_calendar, (event_time, event_type, train_id))

    def dispatch_trains(self):
        """Initialise all trains dispatch time and station"""
        for i,train in enumerate(self.dispatch):
            train_id = i
            t = train[0]
            station = train[1]
            direction = train[2]
            if station not in self.network:
                print(f'dispatch station `{station}` not in network')
                return
            train = Train(train_id, station, direction, t, self.network)
            self.trains[train_id] = train
            self.push(t, 'arrive', train_id)

    def handle_arrive(self, event_time, train_id):
        """Handle a train arrival event — add train to station queue
        and attempt to serve it if the platform is free."""
        train = self.trains[train_id]
        st = self.station_state[train.station][train.direction]

        st['total_arrivals'] += 1
        st['queue'].append((event_time, train_id))

        self._try_serve(train.station, train.direction)

    def _try_serve(self, station, direction):
        """Attempt to serve the next train in the queue at a station.
        Does nothing if the platform is occupied or the queue is empty."""
        st = self.station_state[station][direction]

        if st['occupied_by'] is not None:
            return  # platform busy — train must wait in queue
        if not st['queue']:
            return  # no trains waiting

        arrival_time, train_id = st['queue'].popleft()
        train = self.trains[train_id]

        # Compute how long the train waited in the queue
        wait = self.clock - arrival_time
        st['occupied_by']    = train_id
        st['visit']          += 1
        st['wait_time']      += wait
        st['queue_weighted'] += (len(st['queue']) + 1) * wait  # weighted queue length
        st['frequency']      += self.clock - st['last_departure']

        train.arrival[station].append(self.clock)

        # Sample dwell time — fall back to opposite direction params at terminals
        net = self.network[station]
        d = direction if 'dwell_params' in net[direction] else not direction
        dwell_time = self.sample(**net[d]['dwell_params'])

        self.push(self.clock + dwell_time, 'depart', train_id)

    def handle_depart(self, event_time, train_id):
        """Handle a train departure event — record headway, determine next
        station and direction, sample travel time, and schedule next arrival."""
        train = self.trains[train_id]
        prev_station   = train.station     # save before updating train state
        prev_direction = train.direction
        st = self.station_state[prev_station][prev_direction]

        # Record headway: time since last train departed this station
        train.headway.append(self.clock - st['last_departure'])

        st['occupied_by']   = None
        st['last_departure'] = event_time
        train.departure[prev_station].append(event_time)

        # Determine next direction — flip at terminus if no next station in current direction
        net = self.network[prev_station]
        if 'next_station' in net[prev_direction]:
            next_direction = prev_direction
        else:
            next_direction = not prev_direction   # reverse at terminus

        # Sample travel time to next station and schedule arrival
        next_info   = net[next_direction]['next_station']
        travel_time = self.sample(**next_info['travel_params'])

        train.station   = next_info['name']
        train.direction = next_direction

        self.push(event_time + travel_time, 'arrive', train_id)

        # Free the platform and serve next waiting train
        self._try_serve(prev_station, prev_direction)

    def get_station_stat(self, remark=None):
        """Compile per-station, per-direction summary statistics.
        Skips stations with no visits or no dwell params.
        
        Returns a DataFrame with frequency, avg wait time, and avg queue length."""
        rows = []
        for station in self.network:
            for direction in [True, False]:
                if 'dwell_params' not in self.network[station][direction]:
                    continue
                st = self.station_state[station][direction]
                if st['visit'] == 0:
                    continue
                rows.append({
                    'station':       station,
                    'direction':     direction,
                    'frequency':     round(st['frequency'] / st['visit']),
                    'avg_wait_time': round(st['wait_time'] / st['total_arrivals'], 3) if st['total_arrivals'] else 0,
                    'avg_queue_len': round(st['queue_weighted'] / st['wait_time'], 3) if st['wait_time'] else 0,
                    **(remark or {})
                })
        return pd.DataFrame(rows)

    def get_train_timetable(self, remark=None):
        """Compile full timetable — one row per train visit to each station,
        with arrival and departure times.
        
        Returns a DataFrame sorted by train_id and arrival time."""
        rows = []
        for train_id, train in self.trains.items():
            for station in self.network:
                arrivals   = train.arrival[station]
                departures = train.departure[station]
                for arr, dep in zip(arrivals, departures):
                    rows.append({
                        'train_id':  train_id,
                        'station':   station,
                        'arrival':   arr,
                        'departure': dep,
                        **(remark or {})
                    })
        return pd.DataFrame(rows).sort_values(['train_id', 'arrival']).reset_index(drop=True)

    def get_headway(self, remark=None):
        """Compile headway records — one row per headway observation per train.
        
        Returns a DataFrame with train_id and headway in seconds."""
        rows = []
        for train_id, train in self.trains.items():
            for hw in train.headway:
                rows.append({
                    'train_id': train_id,
                    'headway':  hw,
                    **(remark or {})
                })
        return pd.DataFrame(rows)

    def run(self):
        """Execute the simulation for all replications.
        
        Each replication resets state, dispatches trains, and processes
        events until `until` seconds. Results are accumulated into
        `station_df`, `timetable`, and `train_headway`."""
        for itr in tqdm(range(1, self.repetition + 1)):
            self.fresh_state()
            self.dispatch_trains()

            # Process events in chronological order until end time
            while self.clock <= self.until:
                event_time, event_type, train_id = heapq.heappop(self.event_calendar)
                self.clock = event_time

                if event_type == 'arrive':
                    self.handle_arrive(event_time, train_id)
                elif event_type == 'depart':
                    self.handle_depart(event_time, train_id)

            # Accumulate results from this replication
            self.station_df = pd.concat(
                [self.station_df, self.get_station_stat({'rep': itr})],
                ignore_index=True
            )
            self.timetable = pd.concat(
                [self.timetable, self.get_train_timetable({'rep': itr})],
                ignore_index=True
            )
            self.train_headway = pd.concat(
                [self.train_headway, self.get_headway({'rep': itr})],
                ignore_index=True
            )

In [8]:
blue_network.keys()

dict_keys(['Bowdoin', 'Government Center', 'State', 'Aquarium', 'Maverick', 'Airport', 'Wood Island', 'Orient Heights', 'Suffolk Downs', 'Beachmont', 'Revere Beach', 'Wonderland'])

## run simulation

In [9]:
dispatch = [(0,'Orient Heights',True),      (400,'Orient Heights',True),
            (800,'Orient Heights',True),    (1200,'Orient Heights',True),
            (1600,'Orient Heights',True),   (2000,'Orient Heights',True),]

# rush hour from 9 AM to 2 PM
rush_hour = [{'from': 14400, 'to': 32400, 'params': {'c':0.928, 'd': 1.131,'scale': 0.925}}]

blue_sim = Sim(network=blue_network, until=72000, distribution=scipy.stats.burr,
                dispatch=dispatch, repetition=200, loop_time=2400,
                seed=None)
blue_sim.run()
station_df = blue_sim.station_df

100%|██████████| 200/200 [41:05<00:00, 12.33s/it]   


In [10]:
station_df.groupby(['rep']).agg({'avg_wait_time': 'sum'}).mean().round(2)

avg_wait_time    3.55
dtype: float64

## Output: `station_df`

Stores station-level performance statistics accumulated across all simulation replications.
Each row represents one station–direction pair for one replication.

| Column | Description |
|---|---|
| `station` | Station name |
| `direction` | `True` = outbound, `False` = inbound |
| `frequency` | Mean inter-departure time (seconds) — proxy for headway |
| `avg_wait_time` | Mean time trains spend waiting in the platform queue (seconds) |
| `avg_queue_len` | Mean number of trains queued behind the platform |
| `rep` | Replication number |

In [11]:
station_df

,station,direction,frequency,avg_wait_time,avg_queue_len,rep
0,Bowdoin,False,345,0.144,1.0,1
1,Government Center,True,347,0.211,1.0,1
2,Government Center,False,339,0.056,1.0,1
3,State,True,348,0.168,1.0,1
4,State,False,347,0.056,1.0,1
...,...,...,...,...,...,...
4395,Beachmont,True,351,0.000,0.0,200
4396,Beachmont,False,353,0.000,0.0,200
4397,Revere Beach,True,350,0.044,1.0,200
4398,Revere Beach,False,349,0.000,0.0,200


In [12]:
station_df.to_csv(r'.\data\blue_station_stat.csv')

## Output: `timetable`

Records every train visit to every station across all simulation replications.
Each row represents a single train arriving and departing a station.

| Column | Description |
|---|---|
| `train_id` | Unique train identifier |
| `station` | Station name |
| `arrival` | Simulated arrival time (seconds from simulation start) |
| `departure` | Simulated departure time (seconds from simulation start) |
| `rep` | Replication number |

In [13]:
timetable = blue_sim.timetable
timetable

,train_id,station,arrival,departure,rep
0,0,Orient Heights,0,37,1
1,0,Suffolk Downs,97,148,1
2,0,Beachmont,199,239,1
3,0,Revere Beach,305,349,1
4,0,Wonderland,386,424,1
...,...,...,...,...,...
793247,5,Orient Heights,71277,71324,200
793248,5,Wood Island,71424,71465,200
793249,5,Airport,71527,71588,200
793250,5,Maverick,71719,71782,200


In [14]:
timetable.to_csv(r'.\data\blue_timetable.csv')

## Output: `train_headway`

Records the headway observed at each departure event for every train across all replications.
Each row represents the time gap (in seconds) between the current train and the previous
train to depart the same station.

| Column | Description |
|---|---|
| `train_id` | Unique train identifier |
| `headway` | Time since previous train departed the same station (seconds) |
| `rep` | Replication number |

In [15]:
headway = blue_sim.train_headway
headway

,train_id,headway,rep
0,0,37,1
1,0,148,1
2,0,239,1
3,0,349,1
4,0,424,1
...,...,...,...
793247,5,473,200
793248,5,476,200
793249,5,499,200
793250,5,523,200


In [16]:
headway.to_csv(r'.\data\blue_headway.csv')